# Fine-tune GLM-OCR for Vietnamese Diacritical Marks (v2)

Follows the official guide: [examples/finetune/README.md](https://github.com/zai-org/GLM-OCR/blob/main/examples/finetune/README.md)

**Dataset format:** ShareGPT with `messages`/`role`/`content` + `images` fields.

**Workflow:**
1. Chạy các bước 1→5 (setup, chỉ 1 lần)
2. Bước 6: Train nhấn lại mỗi epoch
3. Bước 7: Merge & eval kết quả
4. Nếu chưa ổn -> quay lại bước 6 train thêm epoch -> chạy lại bước 7
5. Model đã tự lưu trên Drive, tải về khi cần

**Requirements:**
- GPU: T4 (16GB) or better
- Upload `vietnamese_ocr.zip` to Google Drive `My Drive/ocr_data/`
  - Structure: `vietnamese_ocr/vietnamese_ocr.json` + `vietnamese_ocr/images/txt_*.png`

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Mount Drive & Extract Dataset

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p "/content/drive/My Drive/ocr_data"

In [ ]:
# Extract & verify dataset
!cp "/content/drive/My Drive/ocr_data/vietnamese_ocr.zip" /content/
!cd /content && unzip -q -o vietnamese_ocr.zip
!echo "Images:" 0
!echo "JSON:" $(ls -lh /content/vietnamese_ocr/vietnamese_ocr.json)

import json, os

with open("/content/vietnamese_ocr/vietnamese_ocr.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Total samples: {len(data)}")

# Check meta.json
meta_path = "/content/vietnamese_ocr/meta.json"
if os.path.exists(meta_path):
    with open(meta_path) as f:
        meta = json.load(f)
    print(f"Meta: train={meta["num_train"]}, val={meta["num_val"]}, test={meta["num_test"]}")
    assert len(data) >= meta["num_train"] + meta["num_val"] + meta["num_test"], "ERROR: sample count mismatch"
    print(f"OK: {len(data)} >= {meta["num_train"]}+{meta["num_val"]}+{meta["num_test"]}")
else:
    print("⚠️ meta.json not found (older dataset), using defaults")
print("Sample 0:")
print(json.dumps(data[0], ensure_ascii=False, indent=2))

sample = data[0]
assert "messages" in sample, "ERROR: missing messages key"
assert "images" in sample, "ERROR: missing images key"
assert sample["messages"][0]["role"] == "user", "ERROR: first message must be user"
assert sample["messages"][1]["role"] == "assistant", "ERROR: second message must be assistant"
assert "<image>" in sample["messages"][0]["content"], "ERROR: user message must contain <image>"
print("OK: Dataset format is correct!")

img_path = os.path.join("/content/vietnamese_ocr", sample["images"][0])
print(f"Image path: {img_path}")
print(f"Image exists: {os.path.exists(img_path)}")


## 3. Install LLaMA-Factory

In [ ]:
# Install LLaMA-Factory + pin transformers
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd /content/LLaMA-Factory
!pip install -e ".[torch,metrics]" 2>&1 | tail -5

# Pin transformers 5.6.0 — compatible with both LLaMA-Factory (>=4.55, <=5.6.0) and GLM-OCR (>=5.3.0)
!pip install transformers==5.6.0 2>&1 | tail -3

!llamafactory-cli version


## 4. Download GLM-OCR Model

In [ ]:
from huggingface_hub import snapshot_download

model_dir = snapshot_download(
    "zai-org/GLM-OCR",
    local_dir="/content/GLM-OCR",
    local_dir_use_symlinks=False,
)
print(f"Model downloaded to: {model_dir}")

## 5. Prepare Dataset for LLaMA-Factory

Copy dataset into `LLaMA-Factory/data/` (image paths are relative to this directory).

In [ ]:
# Copy dataset into LLaMA-Factory/data/ and register
!cp /content/vietnamese_ocr/vietnamese_ocr.json /content/LLaMA-Factory/data/
!cp -r /content/vietnamese_ocr/images /content/LLaMA-Factory/data/images

import json, os, random

# ── Đọc meta.json để biết số lượng split ──
meta_path = "/content/vietnamese_ocr/meta.json"
if os.path.exists(meta_path):
    with open(meta_path, "r") as f:
        meta = json.load(f)
    N_TEST = meta["num_test"]
    N_VAL = meta["num_val"]
    print(f"Meta: train={meta["num_train"]}, val={N_VAL}, test={N_TEST}, total={meta["total"]}")
else:
    N_TEST = 100
    N_VAL = 100
    print("⚠️ meta.json not found, using default: val=100, test=100")

# ── Split dataset: train / val / test ──
random.seed(42)
with open("/content/LLaMA-Factory/data/vietnamese_ocr.json", "r") as f:
    all_data = json.load(f)

indices = list(range(len(all_data)))
random.shuffle(indices)

test_indices = set(indices[:N_TEST])
val_indices = set(indices[N_TEST:N_TEST + N_VAL])

train_data = [all_data[i] for i in range(len(all_data)) if i not in test_indices and i not in val_indices]
val_data = [all_data[i] for i in sorted(val_indices)]
test_data = [all_data[i] for i in sorted(test_indices)]

# Overwrite train set (no val leak), write separate val/test files
with open("/content/LLaMA-Factory/data/vietnamese_ocr.json", "w") as f:
    json.dump(train_data, f, ensure_ascii=False)
with open("/content/LLaMA-Factory/data/vietnamese_ocr_val.json", "w") as f:
    json.dump(val_data, f, ensure_ascii=False)
with open("/content/LLaMA-Factory/data/vietnamese_ocr_test.json", "w") as f:
    json.dump(test_data, f, ensure_ascii=False)
# Also write test set to /content/ for eval cell
with open("/content/vietnamese_ocr/vietnamese_ocr_test.json", "w") as f:
    json.dump(test_data, f, ensure_ascii=False)

print(f"Split: {len(train_data)} train, {len(val_data)} val, {len(test_data)} test")

# Verify images
missing = 0
for item in train_data + val_data + test_data:
    for img in item["images"]:
        if not os.path.exists(os.path.join("/content/LLaMA-Factory/data", img)):
            missing += 1
print(f"Missing images: {missing}")

# Register datasets
ds_info_path = "/content/LLaMA-Factory/data/dataset_info.json"
with open(ds_info_path, "r") as f:
    info = json.load(f)

for name, fname in [("vietnamese_ocr", "vietnamese_ocr.json"),
                    ("vietnamese_ocr_val", "vietnamese_ocr_val.json"),
                    ("vietnamese_ocr_test", "vietnamese_ocr_test.json")]:
    info[name] = {
        "file_name": fname,
        "formatting": "sharegpt",
        "columns": {"messages": "messages", "images": "images"},
        "tags": {"role_tag": "role", "content_tag": "content", "user_tag": "user", "assistant_tag": "assistant"}
    }

with open(ds_info_path, "w") as f:
    json.dump(info, f, indent=2, ensure_ascii=False)

print(f"OK: registered train ({len(train_data)}) / val ({len(val_data)}) / test ({len(test_data)})")


---

## 6. Train (nhan lai de train them epoch)

Moi lan chay = them 1 epoch. Tu detect checkpoint cu de resume.
- **Lan dau:** train epoch 1 tu dau
- **Lan sau:** resume tu checkpoint cu, train them 1 epoch

Mat khoang 2 phut/epoch tren T4. Checkpoint duoc tu luu len Drive sau khi train xong.

In [ ]:
import os, json

# Detect checkpoint
ckpt_dir = "/content/drive/My Drive/ocr_data/glm-ocr-vn-checkpoints"
checkpoints = sorted([d for d in os.listdir(ckpt_dir) if d.startswith("checkpoint-")], key=lambda x: int(x.split("-")[1])) if os.path.exists(ckpt_dir) else []

if checkpoints:
    last_ckpt = os.path.join(ckpt_dir, checkpoints[-1])
    step = int(checkpoints[-1].split("-")[1])
    with open("/content/LLaMA-Factory/data/vietnamese_ocr.json", "r") as f:
        _ds = json.load(f)
    steps_per_epoch = len(_ds) // (4 * 4)
    next_epoch = step // steps_per_epoch + 1
    total_epochs = next_epoch
    print(f"Resuming from {checkpoints[-1]} (epoch {next_epoch}) -> training to epoch {total_epochs}")
else:
    last_ckpt = None
    total_epochs = 1
    print("No checkpoint found -> training epoch 1")

# Write YAML
yaml_content = f"""
### model
model_name_or_path: /content/GLM-OCR
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 8
lora_alpha: 32
lora_dropout: 0.1
lora_target: all
freeze_vision_tower: true
freeze_multi_modal_projector: false

### dataset
dataset: vietnamese_ocr
eval_dataset: vietnamese_ocr_val
template: glm_ocr
cutoff_len: 2048
preprocessing_num_workers: 8
dataloader_num_workers: 2
eval_strategy: steps
eval_steps: 500
per_device_eval_batch_size: 1

### output
output_dir: "/content/drive/My Drive/ocr_data/glm-ocr-vn-checkpoints"
logging_steps: 10
save_steps: 500
plot_loss: true
overwrite_output_dir: false
save_only_model: false
report_to: none

### train
per_device_train_batch_size: 4
gradient_accumulation_steps: 4
learning_rate: 1.0e-4
num_train_epochs: {total_epochs}
lr_scheduler_type: constant_with_warmup
warmup_ratio: 0.05
fp16: true
"""
if last_ckpt:
    yaml_content += f'resume_from_checkpoint: \"{last_ckpt}\"\n'

with open("/content/glm_ocr_vn_lora_sft.yaml", "w") as f:
    f.write(yaml_content)

os.environ["DISABLE_VERSION_CHECK"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
!llamafactory-cli train /content/glm_ocr_vn_lora_sft.yaml


## 7. Merge & Eval

Sau mỗi epoch, merge LoRA weights vào model gốc và đánh giá trên test set.
Model đã merge được lưu thẳng lên Drive.

In [ ]:
# Merge LoRA weights
import os

ckpt_dir = "/content/drive/My Drive/ocr_data/glm-ocr-vn-checkpoints"
checkpoints = sorted(
    [d for d in os.listdir(ckpt_dir) if d.startswith("checkpoint-")],
    key=lambda x: int(x.split("-")[1])
) if os.path.exists(ckpt_dir) else []

if not checkpoints:
    raise FileNotFoundError(f"No checkpoints found in {ckpt_dir}")

latest = os.path.join(ckpt_dir, checkpoints[-1])
print(f"Merging from {checkpoints[-1]}")

!llamafactory-cli export \
  --model_name_or_path /content/GLM-OCR \
  --adapter_name_or_path "{latest}" \
  --template glm_ocr \
  --export_dir "/content/drive/My Drive/ocr_data/glm-ocr-vn" \
  --trust_remote_code true

In [ ]:
# Install eval dependency
!pip install editdistance -q

import editdistance, json, os, random, glob
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch
from IPython.display import display, HTML

# ── Config ──
# ── Read test count from meta.json ──
_meta = {}
if os.path.exists("/content/vietnamese_ocr/meta.json"):
    with open("/content/vietnamese_ocr/meta.json", "r") as _f:
        _meta = json.load(_f)
EVAL_N_SAMPLES = _meta.get("num_test", 100)
print(f"Eval config: test samples = {EVAL_N_SAMPLES} (from meta.json)")
RESULTS_FILE = "/content/drive/My Drive/ocr_data/glm-ocr-vn-eval-history.json"
TEST_JSON = "/content/vietnamese_ocr/vietnamese_ocr_test.json"
MERGED_DIR = "/content/drive/My Drive/ocr_data/glm-ocr-vn"

# ── Diacritic groups ──
DIACRITIC_GROUPS = {
    "ă (ắằẳẵặ)": set("ăắằẳẵặ"),
    "â (ấầẩẫậ)": set("âấầẩẫậ"),
    "ê (ếềểễệ)": set("êếềểễệ"),
    "ô (ốồổỗộ)": set("ôốồổỗộ"),
    "ơ (ớờởỡợ)": set("ơớờởỡợ"),
    "ư (ứừửữự)": set("ưứừửữự"),
    "đ": set("đĐ"),
}
ALL_DIACRITICS = set().union(*DIACRITIC_GROUPS.values())

def align_chars(gt, pred):
    """Optimal character alignment via edit-distance DP for diacritic evaluation."""
    m, n = len(gt), len(pred)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if gt[i - 1] == pred[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j], dp[i][j - 1], dp[i - 1][j - 1])
    pairs = []
    i, j = m, n
    while i > 0 or j > 0:
        if i > 0 and j > 0 and gt[i - 1] == pred[j - 1]:
            pairs.append((gt[i - 1], pred[j - 1]))
            i -= 1
            j -= 1
        elif i > 0 and j > 0 and dp[i][j] == dp[i - 1][j - 1] + 1:
            pairs.append((gt[i - 1], pred[j - 1]))
            i -= 1
            j -= 1
        elif i > 0 and dp[i][j] == dp[i - 1][j] + 1:
            pairs.append((gt[i - 1], ""))
            i -= 1
        else:
            pairs.append(("", pred[j - 1]))
            j -= 1
    pairs.reverse()
    return pairs

# ── Load model ──
print("Loading merged model...")
processor = AutoProcessor.from_pretrained(MERGED_DIR, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    MERGED_DIR, trust_remote_code=True, torch_dtype="auto", device_map="auto"
)

# ── Load test set ──
if not os.path.exists(TEST_JSON):
    # Fallback: use full dataset, take last 10%
    print("No separate test set found. Using full dataset with val split.")
    TEST_JSON = "/content/vietnamese_ocr/vietnamese_ocr.json"

with open(TEST_JSON, "r", encoding="utf-8") as f:
    test_data = json.load(f)
if EVAL_N_SAMPLES < len(test_data):
    random.seed(42)
    test_data = random.sample(test_data, EVAL_N_SAMPLES)
print(f"Evaluating on {len(test_data)} test samples...")

# ── Run evaluation ──
stats = {
    "cer_total": 0, "char_total": 0,
    "wer_total": 0, "word_total": 0,
    "d_correct": {g: 0 for g in DIACRITIC_GROUPS},
    "d_total": {g: 0 for g in DIACRITIC_GROUPS},
    "exact_match": 0,
}

for i, item in enumerate(test_data):
    gt = item["messages"][1]["content"]
    img_path = os.path.join("/content/vietnamese_ocr", item["images"][0])

    messages = [{"role": "user", "content": [
        {"type": "image", "url": img_path},
        {"type": "text", "text": "Text Recognition:"},
    ]}]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt"
    ).to(model.device)
    inputs.pop("token_type_ids", None)

    generated_ids = model.generate(**inputs, max_new_tokens=512, do_sample=False)
    pred = processor.decode(generated_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    # Exact match
    if pred == gt:
        stats["exact_match"] += 1

    # CER
    stats["cer_total"] += editdistance.eval(pred, gt)
    stats["char_total"] += max(len(gt), 1)

    # WER
    gt_words = gt.split()
    pred_words = pred.split()
    stats["wer_total"] += editdistance.eval(pred_words, gt_words)
    stats["word_total"] += max(len(gt_words), 1)

    # Diacritic accuracy per group (using optimal alignment)
    for group_name, chars in DIACRITIC_GROUPS.items():
        for c_gt, c_pred in align_chars(gt, pred):
            if c_gt in chars:
                stats["d_total"][group_name] += 1
                if c_gt == c_pred:
                    stats["d_correct"][group_name] += 1

    if (i + 1) % 25 == 0:
        print(f"  {i+1}/{len(test_data)} done")

# ── Compute metrics ──
cer = stats["cer_total"] / max(stats["char_total"], 1)
wer = stats["wer_total"] / max(stats["word_total"], 1)
em = stats["exact_match"] / len(test_data) * 100
total_dc = sum(stats["d_correct"].values())
total_dt = sum(stats["d_total"].values())
overall_dacc = total_dc / max(total_dt, 1) * 100

# ── Load/save history ──
history = []
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE, "r") as f:
        history = json.load(f)

current_epoch = len(history) + 1
epoch_result = {"epoch": current_epoch, "n_samples": len(test_data),
                "CER": round(cer, 4), "WER": round(wer, 4),
                "Exact_Match%": round(em, 1), "Diacritic_Acc%": round(overall_dacc, 1)}
for g in DIACRITIC_GROUPS:
    t, c = stats["d_total"][g], stats["d_correct"][g]
    epoch_result[g] = round(c / max(t, 1) * 100, 1) if t > 0 else None

history.append(epoch_result)
os.makedirs(os.path.dirname(RESULTS_FILE), exist_ok=True)
with open(RESULTS_FILE, "w") as f:
    json.dump(history, f, ensure_ascii=False, indent=2)

# ── Display current results ──
print(f"\n{'='*78}")
print(f"  EPOCH {current_epoch} -- {len(test_data)} test samples")
print(f"{'='*78}")
print(f"  CER (Character Error Rate):  {cer:.4f}  càng thấp càng tốt, 0.0 = hoàn hảo")
print(f"  WER (Word Error Rate):       {wer:.4f}  càng thấp càng tốt, 0.0 = hoàn hảo")
print(f"  Exact Match:                 {em:.1f}%   % sample đoán đúng 100% từng ký tự")
print(f"  Diacritic Acc:               {overall_dacc:.1f}%   % dấu TV đoán đúng ({total_dc}/{total_dt})")
print(f"\n  Chi tiết theo từng nhóm nguyên âm (càng cao càng tốt):")
for g in DIACRITIC_GROUPS:
    t, c = stats["d_total"][g], stats["d_correct"][g]
    acc = c / max(t, 1) * 100 if t > 0 else 0
    bar = "█" * int(acc / 5) + "░" * (20 - int(acc / 5))
    print(f"    {g:20s} {acc:5.1f}%  {bar}  ({c}/{t})")

# ── Progress table across epochs ──
if len(history) > 0:
    print(f"\n{'='*78}")
    print(f"  PROGRESS ACROSS ALL EPOCHS")
    print(f"{'='*78}")
    print(f"  Chú thích cột:")
    print(f"    Ep  = Epoch thứ mấy")
    print(f"    CER = Character Error Rate (càng THẤP càng tốt)")
    print(f"    WER = Word Error Rate (càng THẤP càng tốt)")
    print(f"    EM% = Exact Match % (càng CAO càng tốt)")
    print(f"    DA% = Diacritic Accuracy % -- % dấu TV đoán đúng (càng CAO càng tốt)")
    print(f"    ăâêôơưđ = Accuracy riêng từng nhóm nguyên âm kép (càng CAO càng tốt)")
    print()
    
    group_keys = list(DIACRITIC_GROUPS.keys())
    short_names = ["ă", "â", "ê", "ô", "ơ", "ư", "đ"]
    headers = ["Ep", "CER↓", "WER↓", "EM%↑", "DA%↑",] + short_names
    rows = []
    for h in history:
        row = [str(h.get("epoch", "?")),
               f"{h.get('CER',0):.4f}",
               f"{h.get('WER',0):.4f}",
               f"{h.get('Exact_Match%',0):.1f}",
               f"{h.get('Diacritic_Acc%',0):.1f}"]
        for g in group_keys:
            v = h.get(g)
            row.append(f"{v:.1f}" if v is not None else "N/A")
        rows.append(row)
    
    cw = [max(len(r[i]) for r in [headers] + rows) for i in range(len(headers))]
    sep = "+".join(["-" * (w + 2) for w in cw])
    def fmt_row(r):
        return "|" + "|".join(f" {r[i]:<{cw[i]}} " for i in range(len(headers))) + "|"
    
    print(f"  +{sep}+")
    print(f"  {fmt_row(headers)}")
    print(f"  +{sep}+")
    for r in rows:
        print(f"  {fmt_row(r)}")
    print(f"  +{sep}+")

    # Delta vs previous epoch
    if len(history) >= 2:
        prev, curr = history[-2], history[-1]
        print(f"\n  So sánh epoch {prev.get('epoch','?')} -> {curr.get('epoch','?')}:")
        print(f"    [+] = cải thiện  |  [-] = giảm  |  [=] = không đổi")
        display_names = {"CER": "CER (lỗi ký tự)↓", "WER": "WER (lỗi từ)↓",
                         "Exact_Match%": "Exact Match%↑", "Diacritic_Acc%": "Diacritic Acc%↑"}
        for metric in ["CER", "WER", "Exact_Match%", "Diacritic_Acc%"]:
            p, c = prev.get(metric, 0), curr.get(metric, 0)
            diff = c - p
            better = (metric in ["CER", "WER"] and diff < 0) or (metric not in ["CER", "WER"] and diff > 0)
            icon = "[+]" if better else "[-]" if diff != 0 else "[=]"
            name = display_names.get(metric, metric)
            print(f"    {icon} {name:25s} {p} -> {c}  ({'+' if diff>=0 else ''}{diff:.2f})")

print(f"\nResults saved to {RESULTS_FILE}")

**📖 Hướng dẫn đọc bảng kết quả:**

| Cột | Ý nghĩa | Tốt khi |
|-----|---------|----------|
| **CER** | Character Error Rate — tỷ lệ ký tự sai | **Thấp** (0.0 = hoàn hảo) |
| **WER** | Word Error Rate — tỷ lệ từ sai | **Thấp** (0.0 = hoàn hảo) |
| **EM%** | Exact Match — % sample đoán đúng 100% | **Cao** |
| **DA%** | Diacritic Accuracy — % dấu TV đoán đúng | **Cao** (≥95% = tốt) |
| **ă â ê ô ơ ư đ** | Accuracy riêng từng nhóm nguyên âm kép | **Cao** (nhóm nào thấp cần thêm data) |

- [+] = cải thiện so epoch trước | [-] = giảm | [=] = không đổi
- Nếu DA% ≥ 95% -> model đã sẵn sàng trên Drive (`My Drive/ocr_data/glm-ocr-vn/`)
- Nếu DA% < 95% hoặc có nhóm < 80% -> quay **bước 6** train thêm epoch -> chạy lại eval
- Nếu epoch mới toàn [-] -> **overfitting**, dùng checkpoint epoch trước


---

## 8. Test ảnh bất kỳ

Upload 1 ảnh từ máy -> chạy OCR bằng model đã finetune.
Dùng để test nhanh kết quả thực tế sau mỗi epoch.

In [ ]:
# @title Upload & Test 1 ảnh (Original vs Finetuned)
import io
from google.colab import files
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText

ORIG_DIR = "/content/GLM-OCR"
FT_DIR = "/content/drive/My Drive/ocr_data/glm-ocr-vn"

# Upload ảnh
uploaded = files.upload()
fname = list(uploaded.keys())[0]
img = Image.open(io.BytesIO(uploaded[fname])).convert('RGB')
print(f'Đã upload: {fname} ({img.size[0]}x{img.size[1]})')

def run_ocr(proc, mdl, image_path):
    messages = [{'role': 'user', 'content': [
        {'type': 'image', 'url': image_path},
        {'type': 'text', 'text': 'Text Recognition:'},
    ]}]
    inputs = proc.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors='pt'
    ).to(mdl.device)
    inputs.pop('token_type_ids', None)
    ids = mdl.generate(**inputs, max_new_tokens=512, do_sample=False)
    return proc.decode(ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

# Load original model
print('Loading original model...')
orig_proc = AutoProcessor.from_pretrained(ORIG_DIR, trust_remote_code=True)
orig_mdl = AutoModelForImageTextToText.from_pretrained(
    ORIG_DIR, trust_remote_code=True, torch_dtype='auto', device_map='auto'
)

# Load finetuned model (reuse if already loaded from eval)
if 'processor' not in dir() or 'model' not in dir():
    print('Loading finetuned model...')
    processor = AutoProcessor.from_pretrained(FT_DIR, trust_remote_code=True)
    model = AutoModelForImageTextToText.from_pretrained(
        FT_DIR, trust_remote_code=True, torch_dtype='auto', device_map='auto'
    )

# Run both
result_orig = run_ocr(orig_proc, orig_mdl, fname)
result_ft = run_ocr(processor, model, fname)

print(f'\n{"="*50}')
print(f'  ORIGINAL (base)')
print(f'{"="*50}')
print(result_orig)

print(f'\n{"="*50}')
print(f'  FINETUNED (GLM-OCR-VN)')
print(f'{"="*50}')
print(result_ft)

# Word-level diff
if result_orig != result_ft:
    print(f'\n{"="*50}')
    print(f'  DIFF')
    print(f'{"="*50}')
    ow = result_orig.split()
    fw = result_ft.split()
    for i in range(max(len(ow), len(fw))):
        o = ow[i] if i < len(ow) else "<missing>"
        f = fw[i] if i < len(fw) else "<missing>"
        if o != f:
            print(f'  [{i}] "{o}" -> "{f}"')
    if len(ow) != len(fw):
        print(f'  (orig: {len(ow)} words, ft: {len(fw)} words)')
else:
    print('\n  Cả hai giống hệt nhau.')

display(img)
